In [1]:
# @title # Institutional Accumulation Score Calculator (Base)
# =============================================================================
# This script computes the IA_Score for any stock against its sector ETF.
# It fetches hourly data from Yahoo Finance, calculates the five components,
# and outputs the score and each component value.
#
# Usage:
#   1. Run the script.
#   2. Enter the ticker of the stock you want to analyze (e.g., AAPL).
#   3. Enter the ticker of the sector ETF (e.g., XLK for technology).
#   4. Review the output.
# =============================================================================

import pandas as pd
import numpy as np

try:
    import yfinance as yf
except ImportError:
    !pip install yfinance
    import yfinance as yf

# -----------------------------------------------------------------------------
# 1. Fetch hourly data from Yahoo Finance
# -----------------------------------------------------------------------------
def fetch_yahoo_hourly(ticker):
    """
    Fetch hourly OHLCV data for the last 60 days, keep the last 252 completed bars.
    """
    df = yf.download(ticker, period="60d", interval="1h", progress=False, auto_adjust=True)
    if df.empty:
        raise ValueError(f"No data for {ticker}. Please check the ticker symbol.")
    df = df.reset_index()
    df.rename(columns={'Datetime': 'date'}, inplace=True)
    # Timezone handling
    if df['date'].dt.tz is None:
        df['date'] = df['date'].dt.tz_localize('UTC')
    else:
        df['date'] = df['date'].dt.tz_convert('UTC')
    # Drop incomplete current bar
    now = pd.Timestamp.now(tz='UTC')
    if (now - df['date'].iloc[-1]) < pd.Timedelta(hours=1):
        df = df.iloc[:-1]
    # Keep last 252 bars
    df = df.tail(252)
    df = df[['date', 'Open', 'High', 'Low', 'Close', 'Volume']]
    df.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
    df = df.set_index('date')
    return df

# -----------------------------------------------------------------------------
# 2. Compute IA_Score and its components
# -----------------------------------------------------------------------------
def compute_ia_score(asset_df, sector_df):
    """
    Compute the five normalized components and the final IA_Score.
    All components are normalized to [0,1] and averaged.
    """
    # Align indices
    common_idx = asset_df.index.intersection(sector_df.index)
    asset_df = asset_df.loc[common_idx]
    sector_df = sector_df.loc[common_idx]

    # --- CMF (Chaikin Money Flow, 20 periods) ---
    mf = ((asset_df['close'] - asset_df['low']) - (asset_df['high'] - asset_df['close'])) / (asset_df['high'] - asset_df['low'] + 1e-12)
    mf_vol = mf * asset_df['volume']
    cmf = mf_vol.rolling(20, min_periods=10).sum() / asset_df['volume'].rolling(20, min_periods=10).sum()
    cmf_n = (cmf + 1) / 2

    # --- OBV Ratio (OBV / SMA(OBV, 50)) ---
    obv = (np.sign(asset_df['close'].diff()) * asset_df['volume']).fillna(0).cumsum()
    obv_ma = obv.rolling(50, min_periods=10).mean()
    obv_n = (obv / obv_ma).clip(0, 1)

    # --- VWAP Ratio (Close / VWAP, 20 periods) ---
    typical = (asset_df['high'] + asset_df['low'] + asset_df['close']) / 3
    vwap = (typical * asset_df['volume']).rolling(20, min_periods=10).sum() / asset_df['volume'].rolling(20, min_periods=10).sum()
    vwap_n = (asset_df['close'] / vwap).clip(0, 1)

    # --- Sector Alignment (20‑period correlation) ---
    asset_ret = asset_df['close'].pct_change()
    sector_ret = sector_df['close'].pct_change()
    corr = asset_ret.rolling(20, min_periods=10).corr(sector_ret)
    align_n = corr.clip(0, 1)

    # --- Dip Recovery (Bounce volume / Sell-off volume) ---
    n = len(asset_df)
    recovery = pd.Series(index=asset_df.index, dtype=float)
    for i in range(n):
        if i < 9:   # Not enough history
            recovery.iloc[i] = 0.5
            continue
        window = asset_df.iloc[max(0, i-9):i+1]
        trough = window['close'].idxmin()
        tpos = asset_df.index.get_loc(trough)
        if tpos + 5 >= n:
            recovery.iloc[i] = 0.5
            continue
        sell = asset_df['volume'].iloc[max(0, tpos-5):tpos].sum()
        bounce = asset_df['volume'].iloc[tpos+1:tpos+6].sum()
        recovery.iloc[i] = 0.5 if sell == 0 else min(1.0, bounce / sell)

    # --- Final IA_Score (average of five) ---
    ia_score = (cmf_n + obv_n + vwap_n + align_n + recovery) / 5

    return {
        'CMF': cmf_n.iloc[-1],
        'OBV': obv_n.iloc[-1],
        'VWAP': vwap_n.iloc[-1],
        'Alignment': align_n.iloc[-1],
        'Dip_Recovery': recovery.iloc[-1],
        'IA_Score': ia_score.iloc[-1]
    }

# -----------------------------------------------------------------------------
# 3. Main: Prompt user, fetch data, compute, and display results
# -----------------------------------------------------------------------------
def main():
    print("\n=== Institutional Accumulation Score (IA_Score) Calculator ===\n")
    asset_ticker = input("Enter the ticker of the stock to analyze (e.g., AAPL): ").strip().upper()
    sector_ticker = input("Enter the sector ETF ticker (e.g., XLK for technology): ").strip().upper()

    if not asset_ticker or not sector_ticker:
        print("Error: Both tickers are required.")
        return

    try:
        print(f"\nFetching data for {asset_ticker} and {sector_ticker}...")
        asset_df = fetch_yahoo_hourly(asset_ticker)
        sector_df = fetch_yahoo_hourly(sector_ticker)

        components = compute_ia_score(asset_df, sector_df)
        latest_date = asset_df.index[-1]

        title = f"IA_Score for {asset_ticker} (vs {sector_ticker}) as of {latest_date}"
        print("\n" + title)
        print("-" * len(title))
        for key, value in components.items():
            if key == 'IA_Score':
                print(f"\n{key}: {value:.4f}")
            else:
                print(f"{key:>15}: {value:.4f}")
        print("")

        # Optional interpretation hint
        score = components['IA_Score']
        if score >= 0.75:
            print("Interpretation: Very Strong Accumulation – technicals confirm institutional buying.")
        elif score >= 0.50:
            print("Interpretation: Moderate Accumulation – good technicals, but confirm with subjective filters.")
        elif score >= 0.25:
            print("Interpretation: Neutral / Mixed – no clear direction; wait for confirmation.")
        else:
            print("Interpretation: Distribution / Weakness – avoid until conditions improve.")

    except Exception as e:
        print(f"\nError: {e}")
        print("Please check your ticker symbols and try again.")

if __name__ == "__main__":
    main()


=== Institutional Accumulation Score (IA_Score) Calculator ===

Enter the ticker of the stock to analyze (e.g., AAPL): SOXL
Enter the sector ETF ticker (e.g., XLK for technology): SOXX

Fetching data for SOXL and SOXX...

IA_Score for SOXL (vs SOXX) as of 2026-07-17 19:30:00+00:00
-----------------------------------------------------------
            CMF: 0.6225
            OBV: 1.0000
           VWAP: 0.9470
      Alignment: 0.9990
   Dip_Recovery: 0.5137

IA_Score: 0.8164

Interpretation: Very Strong Accumulation – technicals confirm institutional buying.
